### **Student Information**
GitHub ID:

Kaggle name:

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# Project Report 

---
## 1. Model Development

### 1.1 Preprocessing Steps

The competition task is to predict six emotion categories—`anger`, `disgust`, `fear`, `joy`, `sadness`, and `surprise`—based on post text. By merging `final_posts.json`, `data_identification.csv`, and `emotion.csv` on `post_id`, we obtained 3,360 labeled training samples and 1,440 unlabeled test samples. During the merge, `validate="one_to_one"` was used to verify ID alignment; test set labels were not involved in model selection.

We first inspected the training text: 103 samples contained `@` mentions, and no URLs were detected. As seen in the examples, mentions appear as `@<username>`. Since there was no evidence that removing mentions would improve the score, we preserved the original text's casing, punctuation, hashtags, emojis, and mentions. Missing text values were handled using empty strings, and posts were truncated to a maximum of 128 tokens by each model's tokenizer, with dynamic padding applied per batch during training.

Model development utilized a stratified 80/20 split across the six classes (2,688 training, 672 validation, `random_state=42`), evaluated using macro F1 to give equal weight to all six classes in the average. Additionally, a stability check was performed using a fixed setup with a re-applied split of `random_state=2026`. Both splits originated from the same pool of data used for model development and are not treated as fully independent final tests.

### 1.2 Feature Engineering Steps

We initially established a baseline using the TF-IDF method from the class Master notebook: word unigrams/bigrams, followed by comparisons with character 3–5 grams, `char_wb`, class weights, and linear SVM `C` parameters. Next, we compared MiniLM's 384-dimensional sentence vectors with RBF SVM, evaluating model combinations and `disgust` score adjustments via out-of-fold predictions. These experiments were used to establish baselines and understand errors, rather than directly adopting the Master notebook's four-class models; all competition classifiers were retrained on the six-class data.

The final approach uses the respective tokenizers and pretrained weights of `cardiffnlp/twitter-roberta-base-emotion-latest` and `cardiffnlp/twitter-roberta-large-emotion-latest`. The original checkpoints were 11-class multi-label emotion models; we rebuilt them with a six-class single-label output layer and fine-tuned them using the competition training data. Both models output logits for six classes per post, converted to probabilities via softmax, and then combined at a 50/50 ratio. Finally, 0.06 was added to the combined score for `disgust`, and the emotion with the highest score was selected. This addition functions as a classification rule, and the adjusted six numerical values are no longer interpreted as normalized probabilities.

### 1.3 Explanation of Your Model and Evaluation

Both emotion models were trained using a learning rate of `2e-5`, a weight decay of `0.01`, and FP16 for up to 3 epochs, selecting the best checkpoint based on validation macro F1. The per-device batch size was 8 for the base model, and 2 for the large model with gradient accumulation of 4 steps and gradient checkpointing enabled to reduce memory requirements. Once the model and combination rules were finalized, training restarted from the original pretrained weights using all 3,360 labeled samples for 3 epochs to generate the `id,label` CSV for the 1,440 test samples. Prior to submission, we checked row counts, ID uniqueness, missing values, the label set, and the example file's ID order.

| Experiment | Local Validation Macro F1 | Kaggle Public F1 | Decision |
| --- | ---: | ---: | --- |
| Word TF-IDF + Linear SVM Baseline | 0.4898 | 0.4889 | Retained as baseline |
| Word + Char TF-IDF, Balanced Linear SVM,`C=0.1` | 0.5239 | 0.5304 | Included as ensemble member |
| TF-IDF 75%＋MiniLM 25% | 0.5393 | 0.5519 | Continued error analysis |
| Above ensemble with`disgust +0.06`（V4） | 5-fold  OOF 0.5291 | 0.5701 | Best at the time |
| TF-IDF 50%＋MiniLM 50%，`disgust +0.06`（V5） | 5-fold  OOF 0.5404 | 0.5636 | No improvement on Public |
| Added ComplementNB 25%（V6） | OOF ~0.534 at 20%/30% weights; 25% not tested alone | 0.5656 | No improvement on Public |
| Twitter Emotion RoBERTa base, unadjusted | 0.6244 | — | Compared with large ensemble |
| Twitter Emotion RoBERTa large, unadjusted | 0.6260 | — |Compared with base ensemble |
| Emotion base 50% + large 50%, `disgust +0.06` | **0.6568**（seed 42） | 0.6156（trained on holdout validation） | Selected as final architecture |
| Same architecture, retrained on all 3,360 samples| No independent validation set | **0.6370** | Current best Public submission |
| Same architecture and parameters, second stratified split | **0.6480**（seed 2026） | Unsubmitted | Stability check |

The second split only changed data allocation while keeping a fixed 50/50 ratio, `disgust +0.06`, and training settings; the combined score differed from the first split by approximately 0.0088. In the second split, the best validation checkpoints for both base and large occurred at epoch 2; the full-data submission version was trained for 3 epochs per the original fixed settings. This demonstrates that the optimal epoch can vary with the data split, meaning a single validation score should not be treated as a guarantee for the Private score.

Additionally, `microsoft/deberta-v3-base` was previously incorporated into an earlier "General RoBERTa + Twitter RoBERTa" ensemble: DeBERTa weights of 0%, 20%, and 40% yielded validation macro F1 scores of 0.634405, 0.636129, and 0.628211, respectively. A 20% weight improved the score by only about 0.0017, while 40% caused a decrease, so it was excluded from subsequent final ensembles. This comparison does not prove that DeBERTa would be ineffective if added to the final emotion base + large ensemble. Other candidates like BERTweet were also reviewed, but we ultimately chose the emotion-specific model combination which had more thorough validation evidence on the same split.

The Kaggle Public leaderboard evaluates only a subset of the test data; the Private score and final ranking remain unknown. Because models and scores were obtained across different splits and submission versions, local validation scores cannot be treated as submission scores. Repeatedly selecting models using the same validation data can also make results overly optimistic; therefore, the second split was retained as a check, and experiments that yielded no improvement were documented.

### 1.4 AI Prompts, Decisions, and Verification

I used AI to assist in understanding assignment requirements, generating annotated code, suggesting candidate models, and interpreting errors; whether a model was adopted was judged based on my actual execution verification tables, classification reports, and Kaggle submission results. The table below summarizes the primary original prompts and subsequent decisions from our dialogue during the experimental phases; numerical values and code rely on the outputs of this notebook and the Kaggle experimental notebook.

| My Prompt / Request | AI Suggestion & Checks Performed by Me | Decision |
| --- | --- | --- |
| Downloaded competition dataset files, plan to start version 1 with TF-IDF+SVM | Read competition ZIP, verified 3,360/1,440 samples and six emotion classes, established TF-IDF baseline. | Retain original text in this version to avoid accidentally deleting emotion-bearing words, punctuation, or emojis during cleaning. `TfidfVectorizer` converts text into features; subsequent preprocessing or model settings will be adjusted based on validation results. |
| Test what was learned from the Master notebook | Compare character features, class weights, `C`, classifiers, and MiniLM; reuse the same validation split. | Select candidates based on empirically measured F1 rather than directly applying the Master's four-class scores. |
| Stop tuning ensemble weights further and try pretraining on Kaggle Notebooks | AI suggested further fine-tuning pretrained Transformers; testing multiple models on Kaggle T4×2. | Adopted the emotion base + large ensemble with proven validation results. |
| How to determine which model to use | First check if its learned text resembles competition posts, choose principled candidates first. Then compare fairly. Check for complementarity. | Use General RoBERTa as a baseline; Twitter RoBERTa was pretrained on tweets, making it suitable for testing whether tweet phrasing, hashtags, and colloquial expressions help. |
| Continue experiments but compare using validation F1 first before deciding whether to submit | Evaluate RoBERTa, Twitter RoBERTa, DeBERTa, BERTweet, and emotion-specific models on the 2,688/672 stratified split, then retrain the selected approach. | Only submit approaches that have completed validation and submission checks. |
| How to judge whether to continue training | Kaggle data consists of tweets, so Twitter pretrained models may complement general models' judgment. Also tested text processing suggested by the Twitter RoBERTa model card. | Selected the Twitter emotion model combination with higher validation scores. |
| Will the current Public score differ greatly from Private, and how can we verify this? | AI explained Private is unknown; I fixed the final settings, changed only the stratified split seed to 2026, and retrained both models. | The second split score (0.6480) was close to the first (0.6568); used only as stability evidence, not to extrapolate the Private score. |

Error messages, model downloads, GPU settings, and submission format issues were also queried with AI during experiments; these prompts were used to resolve environment issues and were not used as evidence of model performance. The AI did not provide Private labels or scores, and all adoption or rejection decisions were based on saved execution results.

---

# **Competition Code**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/2026-dm-lab-2-private-competition/sampleSubmission.csv
/kaggle/input/competitions/2026-dm-lab-2-private-competition/data_identification.csv
/kaggle/input/competitions/2026-dm-lab-2-private-competition/final_posts.json
/kaggle/input/competitions/2026-dm-lab-2-private-competition/emotion.csv


## 1. Preprocessing Steps

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# 這些路徑只適用於已加入競賽資料的 Kaggle Notebook。
data_dir = Path('/kaggle/input/competitions/2026-dm-lab-2-private-competition')
assert torch.cuda.is_available(), '請先在 Kaggle 開啟 GPU'
with open(data_dir / 'final_posts.json', encoding='utf-8') as f:
    posts = pd.DataFrame(json.load(f))
split = pd.read_csv(data_dir / 'data_identification.csv')
labels = pd.read_csv(data_dir / 'emotion.csv')
sample = pd.read_csv(data_dir / 'sampleSubmission.csv')

data = posts.merge(split, on='post_id', validate='one_to_one')
train = data[data['set'] == 'train'].merge(labels, on='post_id', validate='one_to_one')
test = data[data['set'] == 'test'].copy()

# 與所有候選模型使用相同的分層驗證切分。
train_idx, valid_idx = train_test_split(
    train.index.to_numpy(), test_size=0.2,
    random_state=42, stratify=train['label'],
)
train_part = train.loc[train_idx].reset_index(drop=True)
valid_part = train.loc[valid_idx].reset_index(drop=True)
class_names = sorted(train['label'].unique())
label2id = {name: i for i, name in enumerate(class_names)}
id2label = {i: name for name, i in label2id.items()}
valid_true_ids = valid_part['label'].map(label2id).to_numpy()
print('Train/valid/test:', len(train_part), len(valid_part), len(test))
print('Labels:', class_names)

Train/valid/test: 2688 672 1440
Labels: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


## 2. Feature Engineering Steps

In [2]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

class EmotionDataset(Dataset):
    def __init__(self, frame, tokenizer, with_labels=True):
        # 保留原文，只截斷超過 128 tokens 的貼文；batch 內動態 padding。
        self.encodings = tokenizer(
            frame['text'].fillna('').astype(str).tolist(),
            truncation=True, max_length=128,
        )
        self.labels = (frame['label'].map(label2id).astype(int).tolist()
                       if with_labels else None)

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = self.labels[index]
        return item

base_name = 'cardiffnlp/twitter-roberta-base-emotion-latest'
large_name = 'cardiffnlp/twitter-roberta-large-emotion-latest'
base_tokenizer = AutoTokenizer.from_pretrained(base_name)
large_tokenizer = AutoTokenizer.from_pretrained(large_name)
print('Tokenizers loaded')

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

... (3 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Tokenizers loaded


## 3. Model Implementation Steps

In [3]:
import gc
from scipy.special import softmax
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

def compute_metrics(eval_pred):
    # macro F1 使少數類別在選擇最佳 epoch 時有同等權重。
    logits, true_ids = eval_pred
    return {'macro_f1': f1_score(
        true_ids, np.argmax(logits, axis=-1),
        average='macro', zero_division=0,
    )}

def new_model(model_name):
    # 原模型是 11 類多標籤，重新建立六類單標籤輸出層。
    return AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(class_names),
        id2label=id2label, label2id=label2id,
        problem_type='single_label_classification',
        ignore_mismatched_sizes=True,
    )

base_train_ds = EmotionDataset(train_part, base_tokenizer)
base_valid_ds = EmotionDataset(valid_part, base_tokenizer)
base_test_ds = EmotionDataset(test, base_tokenizer, with_labels=False)
base_model = new_model(base_name)
base_args = TrainingArguments(
    output_dir='/kaggle/working/emotion_base_validation',
    num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    weight_decay=0.01, warmup_steps=50, fp16=True,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1',
    greater_is_better=True, save_total_limit=1,
    logging_strategy='epoch', report_to='none', seed=42, data_seed=42,
)
base_trainer = Trainer(
    model=base_model, args=base_args,
    train_dataset=base_train_ds, eval_dataset=base_valid_ds,
    processing_class=base_tokenizer,
    data_collator=DataCollatorWithPadding(base_tokenizer),
    compute_metrics=compute_metrics,
)
base_trainer.train()
base_valid_probs = softmax(base_trainer.predict(base_valid_ds).predictions, axis=1)
base_test_probs = softmax(base_trainer.predict(base_test_ds).predictions, axis=1)
print('Base validation:', base_trainer.evaluate())
del base_trainer, base_model
gc.collect()
torch.cuda.empty_cache()

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `11`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emotion-latest
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11]) vs model:torch.Size([6])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro F1
1,2.639618,2.158587,0.559624
2,1.808060,2.124451,0.604331
3,1.387602,2.209127,0.619730


... (5 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Macro F1
1.387602,2.209127,3,0.619730


Base validation: {'eval_loss': 2.209127426147461, 'eval_macro_f1': 0.6197303418774509}


In [4]:
# 在同一個 2,688/672 切分上驗證情緒 large 模型。
# 此格位於模型比較之前，確保 large_valid_probs 已建立。
large_train_ds = EmotionDataset(train_part, large_tokenizer)
large_valid_ds = EmotionDataset(valid_part, large_tokenizer)
large_test_ds = EmotionDataset(test, large_tokenizer, with_labels=False)
large_model = new_model(large_name)
large_args = TrainingArguments(
    output_dir='/kaggle/working/emotion_large_validation',
    num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    per_device_eval_batch_size=4, weight_decay=0.01,
    warmup_steps=50, fp16=True, gradient_checkpointing=True,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1',
    greater_is_better=True, save_total_limit=1,
    logging_strategy='epoch', report_to='none', seed=42, data_seed=42,
)
large_trainer = Trainer(
    model=large_model, args=large_args,
    train_dataset=large_train_ds, eval_dataset=large_valid_ds,
    processing_class=large_tokenizer,
    data_collator=DataCollatorWithPadding(large_tokenizer),
    compute_metrics=compute_metrics,
)
large_trainer.train()
large_valid_probs = softmax(
    large_trainer.predict(large_valid_ds).predictions, axis=1
)
print('Large validation:', large_trainer.evaluate())
del large_trainer, large_model
gc.collect()
torch.cuda.empty_cache()

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `11`.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-emotion-latest
Key                        | Status   |                                                                                          
---------------------------+----------+------------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11]) vs model:torch.Size([6])            
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11, 1024]) vs model:torch.Size([6, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[mis

Epoch,Training Loss,Validation Loss,Macro F1
1,10.122967,2.058212,0.616773
2,6.289377,2.084562,0.621562
3,3.918622,2.240089,0.626035


... (5 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Macro F1
3.918622,2.240089,3,0.626035


Large validation: {'eval_loss': 2.2400894165039062, 'eval_macro_f1': 0.6260349879622847}


In [5]:
from sklearn.metrics import precision_recall_fscore_support

# 比較兩個單模型及等權組合；只測先前使用過的 disgust 加值。
candidate_probs = {
    'Emotion base': base_valid_probs,
    'Emotion large': large_valid_probs,
    'Emotion base 50% + large 50%':
        0.5 * base_valid_probs + 0.5 * large_valid_probs,
}
rows = []
for name, probs in candidate_probs.items():
    for bonus in [0.0, 0.06]:
        adjusted = probs.copy()
        adjusted[:, label2id['disgust']] += bonus
        pred_ids = adjusted.argmax(axis=1)
        rows.append({
            'model': name, 'disgust_bonus': bonus,
            'macro_f1': f1_score(valid_true_ids, pred_ids,
                                 average='macro', zero_division=0),
        })
display(pd.DataFrame(rows).sort_values('macro_f1', ascending=False))

# 驗證選定 50/50 與 disgust +0.06；測試集沒有參與選參數。
selected_valid = 0.5 * base_valid_probs + 0.5 * large_valid_probs
selected_valid[:, label2id['disgust']] += 0.06
print(classification_report(
    valid_true_ids, selected_valid.argmax(axis=1),
    target_names=class_names, zero_division=0,
))

,model,disgust_bonus,macro_f1
5,Emotion base 50% + large 50%,0.06,0.632788
4,Emotion base 50% + large 50%,0.00,0.629298
2,Emotion large,0.00,0.626035
3,Emotion large,0.06,0.622293
0,Emotion base,0.00,0.619730
1,Emotion base,0.06,0.616988


              precision    recall  f1-score   support

       anger       0.63      0.67      0.65       135
     disgust       0.49      0.43      0.46        91
        fear       0.65      0.61      0.63        57
         joy       0.74      0.83      0.78       175
     sadness       0.68      0.69      0.69       126
    surprise       0.68      0.52      0.59        88

    accuracy                           0.66       672
   macro avg       0.64      0.63      0.63       672
weighted avg       0.66      0.66      0.66       672



### Final training and submission

The following two models are trained from scratch, starting from the original pre-trained weights, using the full set of 3,360 labeled data points and the hyperparameters fixed during the validation phase. No separate validation set is retained for this stage; the submission files contain predictions solely for the 1,440 unlabeled test data points.

In [6]:
full_base_ds = EmotionDataset(train.reset_index(drop=True), base_tokenizer)
full_base_model = new_model(base_name)
full_base_args = TrainingArguments(
    output_dir='/kaggle/working/full_emotion_base',
    num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=8, weight_decay=0.01,
    warmup_steps=63, fp16=True,
    eval_strategy='no', save_strategy='no',
    logging_strategy='epoch', report_to='none', seed=42, data_seed=42,
)
full_base_trainer = Trainer(
    model=full_base_model, args=full_base_args,
    train_dataset=full_base_ds, processing_class=base_tokenizer,
    data_collator=DataCollatorWithPadding(base_tokenizer),
)
full_base_trainer.train()
full_base_test_probs = softmax(
    full_base_trainer.predict(base_test_ds).predictions, axis=1
)
del full_base_trainer, full_base_model
gc.collect()
torch.cuda.empty_cache()

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `11`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emotion-latest
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11]) vs model:torch.Size([6])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
210,2.526228
420,1.738372
630,1.331744


In [7]:
full_large_ds = EmotionDataset(train.reset_index(drop=True), large_tokenizer)
full_large_model = new_model(large_name)
full_large_args = TrainingArguments(
    output_dir='/kaggle/working/full_emotion_large',
    num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    per_device_eval_batch_size=4, weight_decay=0.01,
    warmup_steps=63, fp16=True, gradient_checkpointing=True,
    eval_strategy='no', save_strategy='no',
    logging_strategy='epoch', report_to='none', seed=42, data_seed=42,
)
full_large_trainer = Trainer(
    model=full_large_model, args=full_large_args,
    train_dataset=full_large_ds, processing_class=large_tokenizer,
    data_collator=DataCollatorWithPadding(large_tokenizer),
)
full_large_trainer.train()
full_large_test_probs = softmax(
    full_large_trainer.predict(large_test_ds).predictions, axis=1
)

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `11`.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-emotion-latest
Key                        | Status   |                                                                                          
---------------------------+----------+------------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11]) vs model:torch.Size([6])            
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([11, 1024]) vs model:torch.Size([6, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[mis

Step,Training Loss
210,9.870655
420,6.199207
630,3.992889


In [8]:
# 將兩個全資料模型的機率等權平均，套用驗證選出的 disgust 加值。
assert full_base_test_probs.shape == full_large_test_probs.shape == (1440, 6)
assert np.allclose(full_base_test_probs.sum(axis=1), 1, atol=1e-5)
assert np.allclose(full_large_test_probs.sum(axis=1), 1, atol=1e-5)
combined = 0.5 * full_base_test_probs + 0.5 * full_large_test_probs
combined[:, label2id['disgust']] += 0.06
pred_ids = combined.argmax(axis=1)
predicted = pd.DataFrame({
    'post_id': test['post_id'].astype(str).to_numpy(),
    'label': [id2label[int(i)] for i in pred_ids],
})

# 依範例檔順序對齊 ID，檢查後寫入 Kaggle working 目錄。
submission = sample[['post_id']].assign(
    post_id=lambda frame: frame['post_id'].astype(str)
).merge(predicted, on='post_id', how='left', validate='one_to_one')
submission = submission.rename(columns={'post_id': 'id'})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test) == 1440
assert submission['id'].is_unique
assert submission.notna().all().all()
assert set(submission['label']).issubset(class_names)
output_path = '/kaggle/working/submission_emotion_base_large_full3360.csv'
submission.to_csv(output_path, index=False)
print(output_path)
print(submission['label'].value_counts())
display(submission.head())

/kaggle/working/submission_emotion_base_large_full3360.csv
label
joy         408
anger       344
sadness     249
surprise    171
disgust     154
fear        114
Name: count, dtype: int64


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise
